# `Structured Output and Pydantic in LangChain`
---

## 1. What is Structured Output?

Normally, an LLM returns **unstructured text**.

For example:

```text
The candidate is Arun. He has 2 years of experience and knows
Python, LangChain and React.
```

This is easy for a human to read, but difficult for an application to process reliably.

A backend application may instead need:

```json
{
  "name": "Arun",
  "experience": 2,
  "skills": ["Python", "LangChain", "React"]
}
```

This is called **structured output**.

### Definition

> **Structured Output** means instructing an LLM to return its response in a predefined structure or schema instead of arbitrary text.

---

# 2. Why Do We Need Structured Output?

LLMs naturally generate text, but applications usually work with structured data.

For example:

### Without Structured Output

```text
The product costs $999 and is available in stock.
```

Your application has to parse this text.

### With Structured Output

```json
{
  "product": "Laptop",
  "price": 999,
  "in_stock": true
}
```

Now your application can directly access:

```python
result["price"]
result["in_stock"]
```

### Common Use Cases

Structured output is useful for:

* Information extraction
* Resume parsing
* Invoice processing
* Classification
* Sentiment analysis
* Product extraction
* Customer-support systems
* RAG applications
* AI agents
* API responses
* Database insertion
* Form filling

---

# 3. What is a Schema?

A **schema** defines what the output should look like.

For example:

```text
Student
├── name → string
├── age → integer
├── course → string
└── skills → list
```

The LLM should produce data that follows this structure.

---

# 4. Structured Output in LangChain

LangChain allows chat models to produce structured results using:

```python
model.with_structured_output(...)
```

For example:

```python
structured_model = model.with_structured_output(MySchema)
```

The schema tells the model:

> "Return your answer according to this structure."

---

# 5. What is Pydantic?

**Pydantic** is a Python library used for **data validation and defining data schemas using Python classes and type hints**.

Example:

```python
from pydantic import BaseModel

class Student(BaseModel):
    name: str
    age: int
    course: str
```

Here we define a schema called `Student`.

It expects:

```text
name   → string
age    → integer
course → string
```

---

# 6. Why Pydantic is Useful with LangChain

Pydantic gives LangChain a clear schema describing the expected output.

For example:

```python
class Student(BaseModel):
    name: str
    age: int
    course: str
```

We can then tell the model:

```python
structured_model = model.with_structured_output(Student)
```

Now the model should return data matching the `Student` schema.

---

# 7. Basic Example

## Step 1: Import Libraries

```python
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
```

---

## Step 2: Create Pydantic Schema

```python
class Student(BaseModel):
    name: str
    age: int
    course: str
```

---

## Step 3: Create Chat Model

```python
model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)
```

---

## Step 4: Enable Structured Output

```python
structured_model = model.with_structured_output(Student)
```

---

## Step 5: Invoke the Model

```python
response = structured_model.invoke(
    "My name is Arun. I am 27 years old and studying MCA."
)
```

Now instead of receiving arbitrary text, we get structured data matching the schema.

Conceptually:

```text
Student(
    name="Arun",
    age=27,
    course="MCA"
)
```

You can access fields directly:

```python
print(response.name)
print(response.age)
print(response.course)
```

---

# 8. Complete Example

```python
from pydantic import BaseModel
from langchain_openai import ChatOpenAI


# Define output schema
class Student(BaseModel):
    name: str
    age: int
    course: str


# Create model
model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


# Add structured output
structured_model = model.with_structured_output(Student)


# Invoke model
response = structured_model.invoke(
    """
    My name is Arun.
    I am 27 years old.
    I am pursuing MCA.
    """
)


# Access structured fields
print("Name:", response.name)
print("Age:", response.age)
print("Course:", response.course)
```

---

# 9. How Does It Work?

The overall flow is:

```text
User Input
    ↓
Pydantic Schema
    ↓
LangChain
    ↓
LLM
    ↓
Structured Response
    ↓
Pydantic Validation
    ↓
Python Object
```

For example:

```text
Input:
"Arun is 27 and pursuing MCA."

        ↓

Schema:
Student
├── name: str
├── age: int
└── course: str

        ↓

LLM

        ↓

Student(
    name="Arun",
    age=27,
    course="MCA"
)
```

---

# 10. Pydantic `BaseModel`

The most important Pydantic concept for LangChain is:

```python
BaseModel
```

Example:

```python
from pydantic import BaseModel

class Product(BaseModel):
    name: str
    price: float
    in_stock: bool
```

This defines the expected structure.

### Fields

```python
name: str
```

means:

> `name` should be a string.

```python
price: float
```

means:

> `price` should be a floating-point number.

```python
in_stock: bool
```

means:

> `in_stock` should be `True` or `False`.

---

# 11. More Complex Pydantic Schema

We can create nested structures.

```python
from pydantic import BaseModel
from typing import List


class Address(BaseModel):
    city: str
    country: str


class Person(BaseModel):
    name: str
    age: int
    skills: List[str]
    address: Address
```

The expected output becomes:

```text
Person
│
├── name
├── age
├── skills
│   ├── skill 1
│   ├── skill 2
│   └── skill 3
│
└── address
    ├── city
    └── country
```

This is useful when extracting complex information from documents.

---

# 12. Practical Application — Resume Information Extractor

Suppose we want to extract candidate information from a resume.

Without structured output:

```text
Arun is a software engineer with experience in React,
Node.js, Python and LangChain. He has worked for 2 years.
```

We need to manually parse this.

Instead, define:

```python
from pydantic import BaseModel
from typing import List


class Candidate(BaseModel):
    name: str
    experience_years: float
    skills: List[str]
```

Then:

```python
structured_model = model.with_structured_output(Candidate)
```

Input:

```python
resume = """
Arun is a software engineer.
He has 2 years of experience.
His skills include React, Node.js, Python and LangChain.
"""
```

Invoke:

```python
candidate = structured_model.invoke(resume)
```

Now:

```python
print(candidate.name)
print(candidate.experience_years)
print(candidate.skills)
```

Conceptually:

```text
Candidate(
    name="Arun",
    experience_years=2,
    skills=[
        "React",
        "Node.js",
        "Python",
        "LangChain"
    ]
)
```

This data can then be inserted into:

* MongoDB
* PostgreSQL
* Elasticsearch
* Vector databases
* APIs
* Search indexes

---

# 13. Pydantic Validation

One major advantage of Pydantic is **validation**.

Suppose the schema says:

```python
class Product(BaseModel):
    name: str
    price: float
```

The application expects:

```json
{
    "name": "Laptop",
    "price": 999.99
}
```

If the returned data doesn't satisfy the schema, Pydantic can detect the problem.

This provides an important layer of reliability between the LLM and your application.

---

# 14. Required vs Optional Fields

You can define required fields:

```python
class User(BaseModel):
    name: str
    age: int
```

Both fields are required.

You can also define optional fields:

```python
from typing import Optional

class User(BaseModel):
    name: str
    age: int
    email: Optional[str] = None
```

Now `email` can be missing.

---

# 15. Field Descriptions

We can provide descriptions for fields.

```python
from pydantic import BaseModel, Field


class Product(BaseModel):

    name: str = Field(
        description="Name of the product"
    )

    price: float = Field(
        description="Current price of the product"
    )

    in_stock: bool = Field(
        description="Whether the product is currently available"
    )
```

These descriptions make the schema more explicit and can help the model understand what each field represents.

---

# 16. Structured Output vs Normal Output

| Normal Output                   | Structured Output       |
| ------------------------------- | ----------------------- |
| Free-form text                  | Predefined structure    |
| Difficult to parse              | Easier to parse         |
| Less predictable                | More predictable        |
| Human-friendly                  | Application-friendly    |
| Manual extraction may be needed | Schema-based extraction |
| Harder to validate              | Can be validated        |

---

# 17. Structured Output vs JSON

These concepts are related but **not exactly the same**.

### JSON

JSON is a **data format**.

```json
{
  "name": "Arun",
  "age": 27
}
```

### Structured Output

Structured output is the broader concept of making an LLM return data according to a predefined schema.

The structured result could be represented using:

* Pydantic model
* JSON
* Typed dictionary
* Other schema definitions

In LangChain, Pydantic is especially convenient because it combines **schema definition + Python typing + validation**.

---

# 18. Pydantic vs Dictionary

### Dictionary

```python
student = {
    "name": "Arun",
    "age": 27
}
```

You can access:

```python
student["name"]
```

But a dictionary doesn't inherently define what fields or types are expected.

### Pydantic

```python
class Student(BaseModel):
    name: str
    age: int
```

Now the structure is explicitly defined and validated.

You can access:

```python
student.name
student.age
```

---

# 19. Structured Output in a LangChain Chain

Structured output can also be combined with prompts.

```python
from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


class Movie(BaseModel):
    title: str
    genre: str
    rating: float


model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

structured_model = model.with_structured_output(Movie)


prompt = ChatPromptTemplate.from_template(
    """
    Extract movie information from the following text:

    {text}
    """
)

chain = prompt | structured_model


response = chain.invoke({
    "text": """
    Inception is a science-fiction movie directed by Christopher Nolan.
    It has an IMDb rating of 8.8.
    """
})

print(response)
```

Conceptual result:

```text
Movie(
    title="Inception",
    genre="Science Fiction",
    rating=8.8
)
```

---

# 20. Structured Output in AI Applications

A very common architecture is:

```text
User / Document
       ↓
Prompt Template
       ↓
LLM
       ↓
Structured Output
       ↓
Pydantic Validation
       ↓
Application Logic
       ↓
Database / API / UI
```

For example, in a resume parser:

```text
Resume
  ↓
LLM
  ↓
Candidate Schema
  ↓
Pydantic Validation
  ↓
Candidate Object
  ↓
MongoDB
```

---

# 21. Important Interview Point: Does Pydantic Make the LLM Deterministic?

**No.**

Pydantic validates the resulting structured data, but it does not turn the LLM itself into a deterministic system.

The LLM still generates the response.

The schema provides constraints and validation around the expected structure.

---

# 22. Important Interview Point: Structured Output Is Not Just Prompting

A weak implementation might say:

```text
Return the answer in JSON.
```

This is just an instruction in a prompt.

Modern structured-output mechanisms can provide stronger schema-based handling, depending on the model/provider and LangChain integration.

Therefore:

```text
"Return JSON"
```

and

```python
model.with_structured_output(MySchema)
```

should not be treated as exactly equivalent approaches.

---

# 23. Important Interview Questions

## Beginner

### 1. What is structured output?

Structured output means getting an LLM response in a predefined schema instead of arbitrary text.

---

### 2. Why is structured output useful?

It makes LLM responses easier for applications to parse, validate, store, and process.

---

### 3. What is Pydantic?

Pydantic is a Python library used to define schemas and validate data using Python type hints.

---

### 4. What is `BaseModel`?

`BaseModel` is the main Pydantic class used to define structured data models.

Example:

```python
class User(BaseModel):
    name: str
    age: int
```

---

### 5. What is `with_structured_output()`?

It configures a supported LangChain chat model to return output according to a specified schema.

```python
structured_model = model.with_structured_output(User)
```

---

# Intermediate

### 6. Why use Pydantic with LangChain?

Pydantic provides a clear schema, Python type information, and validation for structured LLM responses.

---

### 7. What is the difference between JSON and Pydantic?

JSON is a data representation format, whereas Pydantic is a Python library for defining and validating structured data.

---

### 8. Can Pydantic schemas contain nested objects?

Yes.

```python
class Address(BaseModel):
    city: str

class User(BaseModel):
    name: str
    address: Address
```

---

### 9. Can fields be optional?

Yes.

```python
from typing import Optional

class User(BaseModel):
    name: str
    email: Optional[str] = None
```

---

## Scenario-Based

### 10. You need to extract customer information from thousands of emails. Would you use normal text output?

Not ideally. I would define a structured schema using Pydantic and use structured output so each email produces predictable fields that can be validated and stored.

---

### 11. You are building an invoice extraction system. What schema might you define?

For example:

```python
class Invoice(BaseModel):
    invoice_number: str
    customer_name: str
    total_amount: float
    currency: str
```

Then use the schema with the model's structured-output capability.

---

### 12. The LLM sometimes returns missing or incorrectly typed fields. How would you improve the system?

I would:

* Define a precise schema.
* Add meaningful field descriptions.
* Mark required/optional fields correctly.
* Use structured output.
* Validate the result.
* Handle validation failures.
* Add retries or correction logic where appropriate.
* Test the schema against representative inputs.

---

# Key Takeaways

* **Structured Output** means getting LLM responses in a predefined structure.
* LLMs naturally produce free-form text, while applications often need structured data.
* **Pydantic** is a Python library for defining and validating structured data.
* `BaseModel` is used to define Pydantic schemas.
* `with_structured_output()` connects a LangChain chat model to a structured schema.
* Pydantic supports:

  * Type hints
  * Required fields
  * Optional fields
  * Nested models
  * Field descriptions
  * Validation
* Structured output is especially useful for:

  * Information extraction
  * RAG
  * Agents
  * Resume parsing
  * Invoice processing
  * Classification
  * API/database integration

### ⭐ Interview One-Liner

> **Structured output makes an LLM return predictable, schema-based data, while Pydantic provides a Pythonic way to define and validate that schema in LangChain applications.**

### Mental Model

```text
                LLM
                 │
        ┌────────▼────────┐
        │ Structured      │
        │ Output Schema   │
        └────────┬────────┘
                 │
            Pydantic
                 │
        ┌────────▼────────┐
        │ Validated       │
        │ Python Object   │
        └────────┬────────┘
                 │
        Application Logic
                 │
        ┌────────┴────────┐
        ↓                 ↓
    Database             API
```
